In [1]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.5.5+dfsg2-2build1.1).
0 upgraded, 0 newly installed, 0 to remove and 10 not upgraded.


In [3]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [10]:
import subprocess
import time

process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

print("Ollama server started!")

Ollama server started!


In [5]:
!ollama pull llama3.2

In [6]:
!curl http://localhost:11434

Ollama is running

In [7]:
!ollama run llama3.2 "What is artificial intelligence?"

Artificial intelligence (AI) refers to the development of computer systems 
that can perform tasks that would typically require human intelligence, suc
such as learning, problem-solving, decision-making, and perception. These s
systems are designed to mimic the cognitive abilities of humans, allowing t
them to process and analyze large amounts of data, recognize patterns, and 
make predictions or take actions.

AI systems can be broadly classified into two categories:

1. **Narrow or Weak AI**: This type of AI is designed to perform a specific
specific task, such as facial recognition, language translation, or playing
playing chess. Narrow AI systems are trained on a specific dataset and are 
not capable of general reasoning or problem-solving.
2. **General or Strong AI**: This type of AI is designed to perform any int
intellectual task that a human can, including reasoning, problem-solving, a
and learning. General AI systems are still in the early stages of developme
development and a

In [11]:
faq_data = """
College Admission FAQs

Q: What is the admission process?
A: Students must fill out the admission form,
submit required documents and pay admission fees.

Q: What documents are required?
A: 10th marksheet, 12th marksheet, Aadhaar card,
transfer certificate and passport-size photographs.

Q: What is the eligibility criteria?
A: Students must pass 12th standard from a
recognized board with required subjects.

Q: What is the admission fee?
A: The admission fee is Rs. 60000 per year.

Q: Is hostel facility available?
A: Yes, hostel facilities are available.
Contact the college office for details.
"""

print("College FAQ Knowledge Base Created Successfully!")

Advanced College FAQ Knowledge Base Loaded!
Total FAQ categories: 10
College FAQ Knowledge Base Created Successfully!


In [6]:
import subprocess
import time

# Stop any old Ollama process
subprocess.run(["pkill", "-f", "ollama serve"])

time.sleep(2)

# Start Ollama server again
process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(10)

print("Ollama server restarted!")

Ollama server restarted!


In [13]:
!pip install --upgrade gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 36.1 MB/s  0:00:00
  Attempting uninstall: gradio
    Found existing installation: gradio 6.30.0
    Uninstalling gradio-6.30.0:
      Successfully uninstalled gradio-6.30.0


In [ ]:
import gradio as gr
import requests
import time

def chatbot(question):

    if not question.strip():
        return "Please enter a question.", "0 seconds"

    prompt = f"""
    You are a College Admission FAQ Assistant.

    Answer using ONLY the following FAQ data.

    If information is unavailable, say:
    Sorry, this information is not available
    in the college FAQ.

    FAQ DATA:
    {faq_data}

    USER QUESTION:
    {question}
    """

    start_time = time.perf_counter()

    response = requests.post(
        "http://localhost:11434/api/chat",
        json={
            "model": "llama3.2",
            "messages": [
                {"role": "user", "content": prompt}
            ],
            "stream": False
        }
    )

    result = response.json()

    end_time = time.perf_counter()

    inference_time = end_time - start_time

    answer = result["message"]["content"]

    return answer, f"{inference_time:.2f} seconds"


demo = gr.Interface(
    fn=chatbot,
    inputs=gr.Textbox(
        label="Enter Your Question",
        placeholder="Ask about college admission..."
    ),
    outputs=[
        gr.Textbox(label="AI Response"),
        gr.Textbox(label="Inference Time")
    ],
    title="College Admission FAQ Chatbot 🎓",
    description="Ask questions about college admissions using Local AI."
)

demo.launch(
    share=True,
    debug=True,
    server_name="0.0.0.0"
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://7027d4523d88cf25d1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
